# Day 2 Exercise — Website Summarizer with Ollama

**Personal exercise** (fork only, not for upstream PR)

Upgrade the Day 1 website summarizer to use **Ollama locally** via the OpenAI-compatible endpoint from `day2.ipynb`.

Uses our Playwright scraper for JavaScript-heavy sites (e.g. kabbalahmedia.info).

In [1]:
import sys
from pathlib import Path

import requests
from IPython.display import Markdown, display
from openai import OpenAI

# Resolve scraper module whether kernel cwd is repo root or week1/slavapa
_repo_root = Path.cwd()
if (_repo_root / "week1" / "community-contributions" / "slavapa").is_dir():
    _scraper_dir = _repo_root / "week1" / "community-contributions" / "slavapa"
else:
    _scraper_dir = (_repo_root / ".." / "community-contributions" / "slavapa").resolve()

if str(_scraper_dir) not in sys.path:
    sys.path.insert(0, str(_scraper_dir))

from scraper_playwright import fetch_website_contents

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"  # use "llama3.2:1b" on slower machines

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [2]:
# Confirm Ollama is running
requests.get("http://localhost:11434").content

b'Ollama is running'

## Prompts (same pattern as Day 1)

In [3]:
system_prompt = """
You are an assistant that analyzes website contents and provides a short summary.
Ignore navigation menus, footers, and cookie banners.
Respond in markdown. Do not wrap the markdown in a code block.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, summarize these too.

"""

EXERCISE_SITES = [
    "https://kabbalahmedia.info/en/",
    "https://www.michaellaitman.com/",
    "https://kabuconnect.com/",
]

In [4]:
def messages_for(website: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website},
    ]


def summarize(url: str) -> str:
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website),
    )
    return response.choices[0].message.content


def display_summary(url: str) -> None:
    print(f"### {url}\n")
    display(Markdown(summarize(url)))

## Summarize our exercise sites with Ollama (free, local, private)

In [5]:
for site in EXERCISE_SITES:
    display_summary(site)
    print("\n---\n")

### https://kabbalahmedia.info/en/



Kabbalah Media is a website that offers spiritual guidance, teachings, and resources on Kabbalah, a Jewish mystical tradition. The site provides articles, videos, and podcasts on various topics related to Kabbalah, including its history, philosophy, and practical applications.

News/Announcements:
The website currently does not have any active news or announcements section. However, it does publish new content regularly, including articles and videos.


---

### https://www.michaellaitman.com/



**Summary of Website**
This website belongs to Dr. Michael Laitman, a global thinker and expert in education, economics, and social issues. The website covers various topics, including his views on human connection, unity, and the role of education in creating a better world. Dr. Laitman presents his vision for solving global problems through a new method of connection, emphasizing the importance of Jewish unity in combating anti-Semitism and promoting global understanding.

**News and Announcements**
Unfortunately, there are no specific news or announcements on this website, only links to Dr. Laitman's latest publications and media interviews. However, the website does mention some of Dr. Laitman's recent books, including "The Jewish Choice: Unity or Anti-Semitism" and "Completing the Circle".


---

### https://kabuconnect.com/



This website appears to be an online gaming platform that uses the principles of connection and Kabbalah to create a unique and immersive experience for its users. The site offers a variety of games, including rulettis, kolikot, and poker, and promises that players will experience a higher level of happiness, meaning, and purpose through gameplay.

The website also makes claims about the psychological aspects of gaming, suggesting that players will experience adrenaliini-kohottamista, ääniä, ja valoja jo johtuvat tunnostaa voittoon. The platform claims that its games will provide players with a thrilling experience that will keep them coming back for more.

The website features casinos with names like Wildz, Cocoa Casino, and Kanoi Casino, each with its own unique atmosphere and promises to create a new experience for players. The language used is enthusiastic and encouraging, with the aim of enticing new customers to try out the platform.

While there is no explicit mention of news or announcements on the website, the text focused on promoting the gaming platform and its unique experience, leaving little room for discussion of updates or news in this particular content.


---



## Try any URL

In [6]:
display_summary("https://edwarddonner.com")

### https://edwarddonner.com



**Summary**
This website appears to be the homepage of Edward Donner, a co-founder and CTO of AI startup Nebula.io. The website showcases his interests in artificial intelligence, machine learning, and technology. It also mentions his Udemy courses on AI and machine learning, which have been well-received by students.

**News and Announcements**

* January 4, 2026: AI Builder with n8n – Create Agents and Voice Agents – RESOURCES
* September 15, 2025: AI Engineering MLOps Track – Deploy AI to Production – RESOURCES
* May 28, 2025: A general announcement that no specific topic is mentioned, other than an "order" of taking the AI courses.
* February 17, 2026: Vibe Coder to Agentic Engineer – RESOURCES (no specific announcement or topic mentioned)